# Determining electric conductivity $\sigma$ and magnetic field $B$

### Imports

In [ ]:

from pathlib import Path
import numpy as np
import warnings


import matplotlib.pyplot as plt


from ppvm_py.utils import load_object

from ppvm_py.plotting import plot_heatmap
from ppvm_py.plotting import generate_2d_video

from ppvm_py.data_processing.patt_fld8v import print_xyz_slice_example_from_parsed_data
from ppvm_py.data_processing.utils import create_index_to_coord_map

from ppvm_py.feature_generation.sdc import compute_symmetric_difference_coefficients_with_index_to_coord_map

### General Args

In [ ]:
# IMPORTANT: Restart Kernel when swapping datasets to reload them!
Ha = 1000

if Ha == 300:
    patt_fld8v_folder_path = Path("../../../../data/4pi_re1000_ha300.360_ch28.p1/patt_fld8v/")
elif Ha == 1000:
    patt_fld8v_folder_path = Path("../../../../data/4pi_re1000_ha1000.384.pi/patt_fld8v/")

fld_path = patt_fld8v_folder_path / Path("pkl/patt_fld8v.pkl")

# Estimates from evaluations
sigma = -1  # According to the results here.
B_z = -1  # According to the results here.

animation_folder = Path("output/animations/sigma_and_b/")

truncate_data = True
truncate_at_snapshot = 10  # Including indices 0 to (truncate_at_snapshot - 1)

### Load Data

In [3]:
if 'fld_data' not in locals():
    fld_data = load_object(fld_path)
    if truncate_data:
        fld_data['timeseries'] = fld_data['timeseries'][:truncate_at_snapshot]
    fld_data['index_to_coord_map'] = create_index_to_coord_map(fld_data['coord_to_index_map'])
print_xyz_slice_example_from_parsed_data(fld_data)

Object successfully loaded from C:\Users\hydro\Documents\PhD\nuclear_fusion_cooling\data\4pi_re1000_ha1000.384.pi\patt_fld8v\pkl\patt_fld8v.pkl
The 3D data looks as follows:
N (number of snapshots) = 10
n_x (grid depth) = 577, n_y (grid height) = 97, n_z (grid width) = 97, n_v (number of variables) = 8
Labels:
['vx', 'vy', 'vz', 'jx', 'jy', 'jz', 'P', 'F']
Preview of the first snapshot (first 2 x, first 2 y, first 2 z) with coordinates:
  (x=0.00, y=-1.00, z=-1.00): [ 0.          0.          0.          0.          0.          0.
  0.00070366 -0.1506608 ]
  (x=0.00, y=-1.00, z=-0.99): [ 0.          0.          0.          0.          0.         -0.01773967
  0.00071547 -0.1506508 ]
  (x=0.00, y=-0.99, z=-1.00): [ 0.         0.         0.         0.         0.3698478  0.
  0.0007184 -0.1515782]
  (x=0.00, y=-0.99, z=-0.99): [ 0.          0.          0.          0.          0.4032355  -0.00119388
  0.00074057 -0.1516712 ]
  (x=0.02, y=-1.00, z=-1.00): [ 0.000000e+00  0.000000e+00  0.0000

### Ohm's Law

$\mathbf{J} = \sigma (\mathbf{E} + \mathbf{v} \times \mathbf{B})$

In vector form:

$\mathbf{J} = \begin{pmatrix} j_x \\ j_y \\ j_z \end{pmatrix} = \sigma \begin{pmatrix} E_x + (v_y B_z - v_z B_y) \\ E_y + (v_z B_x - v_x B_z) \\ E_z + (v_x B_y - v_y B_x) \end{pmatrix}$

We know that $B$ is only present in $z$-direction.

$\begin{pmatrix} j_x \\ j_y \\ j_z \end{pmatrix} = \sigma \left(\begin{pmatrix}E_x \\ E_y \\ E_z \end{pmatrix} + \begin{pmatrix}v_y B_z \\ -v_x B_z \\ 0 \end{pmatrix}\right)$

$= \sigma \left(\begin{pmatrix} \frac{\partial \varphi}{\partial x} \\ \frac{\partial \varphi}{\partial y} \\ \frac{\partial \varphi}{\partial 
z} \end{pmatrix} + \begin{pmatrix}v_y B_z \\ -v_x B_z \\ 0 \end{pmatrix}\right)$

$\approx \sigma \left(\begin{pmatrix} \frac{\Delta \varphi_x}{\Delta l_x} \\ \frac{\Delta \varphi_y}{\Delta l_y} \\ \frac{\Delta \varphi_z}{\Delta l_z} \end{pmatrix} + \begin{pmatrix}v_y B_z \\ -v_x B_z \\ 0 \end{pmatrix}\right)$

$= \sigma \left(\begin{pmatrix} \frac{\varphi_{x_2} - \varphi_{x_1}}{\Delta l_x} \\ \frac{\varphi_{y_2} - \varphi_{y_1}}{\Delta l_y} \\ \frac{\varphi_{z_2} - \varphi_{z_1}}{\Delta l_z} \end{pmatrix} + \begin{pmatrix}v_y B_z \\ -v_x B_z \\ 0 \end{pmatrix}\right)$

So, we have:

$ v_y \approx \frac{1}{B_z} \left( \frac{j_x}{\sigma} - \frac{\varphi_{x_2} - \varphi_{x_1}}{\Delta l_x} \right) $

$ v_x \approx \frac{1}{B_z} \left( \frac{\varphi_{y_2} - \varphi_{y_1}}{\Delta l_y} - \frac{j_y}{\sigma} \right) $

$ \frac{j_z}{\sigma} \approx \frac{\varphi_{z_2} - \varphi_{z_1}}{\Delta l_z}$

## Ohm's Law in z-coord

### Estimate $\frac{\partial{\phi}}{\partial{z}}$

In [4]:
part_phi_z = compute_symmetric_difference_coefficients_with_index_to_coord_map(
    fld_data['timeseries'][..., fld_data['labels'].index('F')],
    direction_axis=3,
    index_to_coord_map=fld_data['index_to_coord_map'],
)

non_zero_data_indices = (
    slice(None),
    slice(None),
    slice(1, -1),
    slice(1, -1),
)  # Exclude first and last y and z index (sdc can not be calculated there).
part_phi_z = part_phi_z[non_zero_data_indices]

### Find 0 Entries

##### $\frac{\partial{\phi}}{\partial{z}}$

In [ ]:
# i_start = 200
# generate_2d_video(
#     data = np.all(
#         part_phi_z[:, i_start:, ...] == 0,
#         axis=0,
#         ),
#     output_file_path=animation_folder / Path("part_phi_z_zeros.gif"),
#     fps=10,
#     xlabel="Y Axis",
#     ylabel="Z Axis",
#     title_pattern="X Index: " + f"{i_start} + " + " {}" if i_start != 0 else "X Index: {}",
# )

##### $j_z$

In [ ]:
j_z = fld_data['timeseries'][
    :,
    :,
    :,
    :,
    fld_data['labels'].index('jz'),
]
j_z = j_z[non_zero_data_indices]

# i_start = 0
# generate_2d_video(
#     data = np.all(
#         j_z == 0,
#         axis=0,
#         ),
#     output_file_path=animation_folder / Path("j_z_zeros.gif"),
#     fps=10,
#     xlabel="Y Axis",
#     ylabel="Z Axis",
#     title_pattern="X Index: " + f"{i_start} + " + " {}" if i_start != 0 else "X Index: {}",
# )

### Calculate Sigma

In [7]:
part_phi_z_non_zero_mask = (part_phi_z != 0)

sigma_est = np.divide(
    j_z,
    part_phi_z,
    out=np.full_like(part_phi_z, np.nan),
    where=part_phi_z_non_zero_mask,
)

##### Mean

In [8]:
# Some slices along axis 0 are nan, therefore a warning is always printed.
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)

    mean = np.nanmean(
        sigma_est,
        axis=0,
    )
mean_mean = np.nanmean(
    mean,
)
print(f"mean(mean(sigma_est)) = {mean_mean}")

# nan_representative = np.nanmax(mean)
# mean_non_nan = mean.copy()
# mean_non_nan[np.isnan(mean)] = nan_representative
# i_start = 0
# generate_2d_video(
#     data=mean_non_nan,
#     output_file_path=animation_folder / Path("sigma_z_mean.gif"),
#     fps=10,
#     xlabel="Y Axis",
#     ylabel="Z Axis",
#     title_pattern="X Index: " + f"{i_start} + " + " {}" if i_start != 0 else "X Index: {}",
#     vmin=-2,
#     vmax=2,
# )

mean(mean(sigma_est)) = -0.9182130604227403


##### 3 Std Dev

In [9]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)

    std = np.nanstd(
        sigma_est,
        axis=0,
    )
std3 = 3 * std
mean_std3 = np.nanmean(
    std3,
)
print(f"mean(3 * std(sigma_est)) = {mean_std3}")

# nan_representative = np.nanmax(std3)
# std3_non_nan = std3.copy()
# std3_non_nan[np.isnan(std3)] = nan_representative
# i_start = 0
# generate_2d_video(
#     data=std3_non_nan,
#     output_file_path=animation_folder / Path("sigma_z_3_std_dev.gif"),
#     fps=10,
#     xlabel="Y Axis",
#     ylabel="Z Axis",
#     title_pattern="X Index: " + f"{i_start} + " + " {}" if i_start != 0 else "X Index: {}",
#     vmin=-2,
#     vmax=2,
# )

mean(3 * std(sigma_est)) = 0.22549388021886246
Video saved to
C:\Users\hydro\Documents\PhD\nuclear_fusion_cooling\potential_probe_velocity_measuring\potential_probe_velocity_measuring\notebooks\3d\animations\sigma_z_3_std_dev.gif


##### Median

In [10]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)

    median = np.nanmedian(
        sigma_est,
        axis=0,
    )
median_median = np.nanmedian(
    median,
)
print(f"median(median(sigma_est)) = {median_median}")

# nan_representative = np.nanmax(median)
# median_non_nan = median.copy()
# median_non_nan[np.isnan(median)] = nan_representative
# i_start = 0
# generate_2d_video(
#     data=median_non_nan,
#     output_file_path=animation_folder / Path("sigma_z_median.gif"),
#     fps=10,
#     xlabel="Y Axis",
#     ylabel="Z Axis",
#     title_pattern="X Index: " + f"{i_start} + " + " {}" if i_start != 0 else "X Index: {}",
#     vmin=-2,
#     vmax=2,
# )

median(median(sigma_est)) = -0.9998195134836642
Video saved to
C:\Users\hydro\Documents\PhD\nuclear_fusion_cooling\potential_probe_velocity_measuring\potential_probe_velocity_measuring\notebooks\3d\animations\sigma_z_median.gif


## Ohm's Law in x-coord

Non worth the effort, the previous sections fits the slice results already and is enough evidence for my cause.

## Ohm's Law in y-coord

Might be interesting to look upon, but I do not want to spend even more time here.